# 12장. TF, IDF, TF-IDF의 수식

| 항목 | 내용 |
|---|---|
| 예상 시간 | 4시간 |
| 선수 장 | 3장, 11장 |
| 이 장의 결과물 | 미니 코퍼스 TF-IDF 손계산 표, README에 넣을 수식 정리 |
| 관련 학습 목표 | ② TF, IDF, TF-IDF의 수학적 정의를 수식으로 표현하고 각각의 역할을 설명 |

**학습 목표**
- TF, DF, IDF, TF-IDF를 수식으로 쓰고 말로 설명할 수 있다.
- raw TF와 sublinear TF의 차이를 숫자로 보여 줄 수 있다.
- smooth IDF 식의 `+1` 세 개가 각각 무슨 역할인지 설명할 수 있다.
- L2 정규화가 왜 필요한지 설명할 수 있다.
- 미니 코퍼스의 TF-IDF를 처음부터 끝까지 손으로 계산할 수 있다.

이 장은 실데이터를 쓰지 않고 **미니 코퍼스 5개 문서**로만 계산합니다. 손계산 결과는 13장에서 구현을 검증하는 정답이 됩니다.

In [1]:
# 미니 코퍼스: 모든 장에서 함께 쓰는 문서 5개 (0장 0.5절 참고)
MINI_DOCS = [
    "The rocket launched into space!",
    "NASA launched a new space telescope.",
    "The pitcher threw a fast ball.",
    "Our team won the baseball game.",
    "Astronauts watched the baseball game in space.",
]
MINI_LABELS = ["space", "space", "baseball", "baseball", "space"]

import numpy as np
import pandas as pd

STOP = {"the", "a", "into", "our", "in", "new"}
FILTERED = [[t for t in doc.lower().strip(".!").split() if t not in STOP] for doc in MINI_DOCS]
np.set_printoptions(precision=4, suppress=True, linewidth=120)

from collections import Counter

VOCAB = sorted({t for doc in FILTERED for t in doc})
N = len(FILTERED)
COUNTS = np.array([[doc.count(w) for w in VOCAB] for doc in FILTERED], dtype=float)
for i, doc in enumerate(FILTERED):
    print(i, doc)
print("어휘", len(VOCAB), "개:", VOCAB)

0 ['rocket', 'launched', 'space']
1 ['nasa', 'launched', 'space', 'telescope']
2 ['pitcher', 'threw', 'fast', 'ball']
3 ['team', 'won', 'baseball', 'game']
4 ['astronauts', 'watched', 'baseball', 'game', 'space']
어휘 15 개: ['astronauts', 'ball', 'baseball', 'fast', 'game', 'launched', 'nasa', 'pitcher', 'rocket', 'space', 'team', 'telescope', 'threw', 'watched', 'won']


## 12.1 직관: 무엇이 중요한 단어인가

문서를 잘 설명하는 단어는 두 조건을 만족합니다.
1. **이 문서에 자주 나온다** → TF(Term Frequency)가 높다
2. **다른 문서에는 드물게 나온다** → IDF(Inverse Document Frequency)가 높다

`the`는 조건 1은 만족하지만 조건 2를 만족하지 못합니다. 모든 문서에 나오기 때문입니다. `telescope`는 doc 1에만 나오므로 doc 1을 다른 문서와 구별하는 좋은 단서입니다. **TF-IDF = TF × IDF**는 두 조건을 곱해서 둘 다 만족하는 단어에 큰 가중치를 줍니다.

## 12.2 TF: 문서 안의 단어 빈도

$$\mathrm{TF}(t, d) = f_{t,d} \quad (\text{문서 } d \text{에서 단어 } t \text{가 나온 횟수})$$

11장의 카운트 행렬이 바로 raw TF입니다.

> **횟수인가, 비율인가?** 교과서에 따라 $f_{t,d} / |d|$(문서 길이로 나눈 비율)로 정의하기도 합니다. sklearn과 이 과제는 **횟수 그대로** 씁니다. 어차피 마지막에 L2 정규화를 하면 문서 길이로 나눈 효과가 사라지기 때문입니다. 아래에서 확인합니다.

In [2]:
df = (COUNTS > 0).sum(axis=0)
idf = np.log((1 + N) / (1 + df)) + 1


def l2(M):
    return M / np.linalg.norm(M, axis=1, keepdims=True)


by_count = l2(COUNTS * idf)
by_ratio = l2((COUNTS / COUNTS.sum(axis=1, keepdims=True)) * idf)
print("횟수 TF와 비율 TF의 최종 결과가 같은가:", np.allclose(by_count, by_ratio))

횟수 TF와 비율 TF의 최종 결과가 같은가: True


행 하나에 같은 수(문서 길이의 역수)를 곱해도 L2 정규화 뒤에는 방향만 남으므로 결과가 같습니다(3장: 정규화는 길이를 지운다).

## 12.3 sublinear TF: 반복의 효과 줄이기

같은 단어가 10번 나왔다고 1번보다 10배 중요할까요? 대개는 아닙니다. 그래서 횟수에 로그를 씌워 증가를 완만하게 만듭니다. sklearn에서는 `sublinear_tf=True`입니다.

$$\mathrm{TF}_{\log}(t, d) = \begin{cases} 1 + \ln f_{t,d} & f_{t,d} > 0 \\ 0 & f_{t,d} = 0 \end{cases}$$

미니 코퍼스는 모든 단어가 문서마다 한 번씩만 나와서 두 방식의 차이가 없습니다($1 + \ln 1 = 1$). 차이를 보려고 반복이 있는 예를 씁니다.

In [3]:
rows = []
for f in [0, 1, 2, 3, 5, 10, 50]:
    rows.append({"횟수 f": f, "raw TF": f, "1 + ln f": round(1 + np.log(f), 3) if f > 0 else 0.0})
pd.DataFrame(rows).set_index("횟수 f").T

횟수 f,0,1,2,3,5,10,50
raw TF,0.0,1.0,2.000,3.000,5.000,10.000,50.000
1 + ln f,0.0,1.0,1.693,2.099,2.609,3.303,4.912


50번 나온 단어가 raw TF로는 1번의 50배지만, sublinear TF로는 약 4.9배입니다. 같은 단어를 반복하는 긴 글이 한두 단어에 지배되는 것을 막아 줍니다. 이 과제의 **기본값은 raw TF**이고, sublinear는 14장 검증의 추가 설정으로 확인합니다.

## 12.4 DF와 IDF

**DF(Document Frequency)**: 단어 $t$가 한 번 이상 나온 **문서 수**입니다(2장에서 `Counter(set(tokens))`로 셌던 것).

**IDF**의 기본 아이디어는 $\dfrac{n}{\mathrm{df}(t)}$, 즉 "전체 문서 중 몇 분의 1에만 나오나"입니다. 드물수록 큽니다. 여기에 **로그**를 씌웁니다(3장).

$$\mathrm{IDF}_{\text{기본}}(t) = \ln \frac{n}{\mathrm{df}(t)}$$

로그를 씌우는 이유: 문서 1,000개에서 df=1인 단어와 df=100인 단어의 비율 차이는 100배인데, 로그를 씌우면 약 1.5배(6.91 / 4.61)로 줄어듭니다. 희귀한 단어 하나가 다른 모든 단어를 압도하지 않게 합니다.

In [4]:
table = pd.DataFrame({"df": df, "n/df": N / df, "ln(n/df)": np.log(N / df)}, index=VOCAB)
table.round(4).T

,astronauts,ball,baseball,fast,game,launched,nasa,pitcher,rocket,space,team,telescope,threw,watched,won
df,1.0000,1.0000,2.0000,1.0000,2.0000,2.0000,1.0000,1.0000,1.0000,3.0000,1.0000,1.0000,1.0000,1.0000,1.0000
n/df,5.0000,5.0000,2.5000,5.0000,2.5000,2.5000,5.0000,5.0000,5.0000,1.6667,5.0000,5.0000,5.0000,5.0000,5.0000
ln(n/df),1.6094,1.6094,0.9163,1.6094,0.9163,0.9163,1.6094,1.6094,1.6094,0.5108,1.6094,1.6094,1.6094,1.6094,1.6094


## 12.5 smooth IDF와 세 개의 +1 ⭐

sklearn의 기본값(`smooth_idf=True`)이자 이 과제의 기본 정의는 다음과 같습니다.

$$\mathrm{IDF}(t) = \ln \frac{1 + n}{1 + \mathrm{df}(t)} + 1$$

`smooth_idf=False`일 때는 다음과 같습니다.

$$\mathrm{IDF}_{\text{no-smooth}}(t) = \ln \frac{n}{\mathrm{df}(t)} + 1$$

| `+1`의 위치 | 역할 |
|---|---|
| 분자 $1 + n$, 분모 $1 + \mathrm{df}$ (smooth) | **모든 단어를 한 번씩 포함하는 가상의 문서가 하나 더 있다**고 보는 것. $\mathrm{df} = 0$인 단어(어휘를 외부에서 정했을 때)도 0으로 나누지 않는다. 값이 조금 완만해진다 |
| 맨 끝의 $+1$ | **모든 문서에 나오는 단어도 가중치가 0이 되지 않게** 한다. $\ln 1 = 0$이므로 +1이 없으면 그런 단어는 완전히 사라진다 |

In [5]:
idf_smooth = np.log((1 + N) / (1 + df)) + 1
idf_plain = np.log(N / df) + 1
idf_no_tail = np.log((1 + N) / (1 + df))          # 끝의 +1이 없다면

table = pd.DataFrame({"df": df, "smooth IDF": idf_smooth, "no-smooth IDF": idf_plain,
                      "끝 +1 없음": idf_no_tail}, index=VOCAB)
table.drop_duplicates().round(4)

,df,smooth IDF,no-smooth IDF,끝 +1 없음
astronauts,1,2.0986,2.6094,1.0986
baseball,2,1.6931,1.9163,0.6931
space,3,1.4055,1.5108,0.4055


미니 코퍼스에는 df가 1, 2, 3인 단어만 있어서 값이 세 종류뿐입니다. 위 표는 같은 값을 한 줄씩만 남겼습니다.

**모든 문서에 나오는 단어**는 어떻게 될까요? 가상의 단어 $\mathrm{df} = n = 5$를 넣어 봅니다.

In [6]:
d = N
print(f"df = n = {N}인 단어")
print(f"  smooth IDF      : ln(6/6) + 1 = {np.log((1 + N) / (1 + d)) + 1:.4f}")
print(f"  no-smooth IDF   : ln(5/5) + 1 = {np.log(N / d) + 1:.4f}")
print(f"  끝 +1 없는 IDF : ln(6/6)     = {np.log((1 + N) / (1 + d)):.4f}  ← 가중치 0, 벡터에서 사라짐")

df = n = 5인 단어
  smooth IDF      : ln(6/6) + 1 = 1.0000
  no-smooth IDF   : ln(5/5) + 1 = 1.0000
  끝 +1 없는 IDF : ln(6/6)     = 0.0000  ← 가중치 0, 벡터에서 사라짐


## 12.6 TF × IDF

$$\mathrm{tfidf}(t, d) = \mathrm{TF}(t, d) \times \mathrm{IDF}(t)$$

문서-단어 행렬로 보면 **각 열(단어)에 그 단어의 IDF를 곱하는** 것입니다. 4장 브로드캐스팅 `(n, V) * (V,)` 그대로입니다. doc 0으로 계산해 봅니다.

In [7]:
i = 0
cols = np.nonzero(COUNTS[i])[0]
step = pd.DataFrame({"TF": COUNTS[i, cols], "df": df[cols], "IDF": idf[cols],
                     "TF×IDF": COUNTS[i, cols] * idf[cols]}, index=[VOCAB[j] for j in cols])
print("doc 0:", FILTERED[0])
step.round(4)

doc 0: ['rocket', 'launched', 'space']


,TF,df,IDF,TF×IDF
launched,1.0,2,1.6931,1.6931
rocket,1.0,1,2.0986,2.0986
space,1.0,3,1.4055,1.4055


`rocket`은 doc 0에만 나와서(df=1) IDF가 가장 크고, `space`는 3개 문서에 나와서(df=3) 가장 작습니다. 같은 TF=1이어도 **드문 단어가 더 큰 가중치**를 받습니다.

## 12.7 L2 정규화

$$\mathbf{x}_d = \frac{\mathbf{v}_d}{\lVert \mathbf{v}_d \rVert_2}, \qquad \mathbf{v}_d = \big(\mathrm{tfidf}(t_1, d), \dots, \mathrm{tfidf}(t_V, d)\big)$$

긴 문서는 단어 횟수가 많아 벡터가 길어지고, 그대로 내적하면 긴 문서가 무조건 유리합니다(3장). 길이를 1로 맞추면 **단어 구성(방향)만** 비교하게 됩니다. 정규화된 벡터끼리의 내적은 곧 코사인 유사도입니다.

In [8]:
v = step["TF×IDF"].to_numpy()
norm = np.sqrt((v ** 2).sum())
print("제곱:", (v ** 2).round(4), "합:", round((v ** 2).sum(), 4))
print(f"‖v‖ = √{(v ** 2).sum():.4f} = {norm:.4f}")
step["정규화"] = v / norm
print("정규화 후 길이:", round(np.sqrt((step['정규화'] ** 2).sum()), 6))
step.round(4)

제곱: [2.8667 4.4042 1.9753] 합: 9.2463
‖v‖ = √9.2463 = 3.0408
정규화 후 길이: 1.0


,TF,df,IDF,TF×IDF,정규화
launched,1.0,2,1.6931,1.6931,0.5568
rocket,1.0,1,2.0986,2.0986,0.6902
space,1.0,3,1.4055,1.4055,0.4622


## 12.8 미니 코퍼스 전체 손계산

doc 0에서 한 계산을 모든 문서에 적용한 최종 TF-IDF 행렬입니다. 0이 아닌 값만 문서별로 보여 줍니다.

In [9]:
TFIDF = l2(COUNTS * idf)
for i in range(N):
    nz = {VOCAB[j]: round(TFIDF[i, j], 4) for j in np.nonzero(TFIDF[i])[0]}
    print(f"doc {i}: {nz}")

doc 0: {'launched': np.float64(0.5568), 'rocket': np.float64(0.6902), 'space': np.float64(0.4622)}
doc 1: {'launched': np.float64(0.4583), 'nasa': np.float64(0.568), 'space': np.float64(0.3804), 'telescope': np.float64(0.568)}
doc 2: {'ball': np.float64(0.5), 'fast': np.float64(0.5), 'pitcher': np.float64(0.5), 'threw': np.float64(0.5)}
doc 3: {'baseball': np.float64(0.444), 'game': np.float64(0.444), 'team': np.float64(0.5503), 'won': np.float64(0.5503)}
doc 4: {'astronauts': np.float64(0.5164), 'baseball': np.float64(0.4166), 'game': np.float64(0.4166), 'space': np.float64(0.3458), 'watched': np.float64(0.5164)}


**읽는 법**
- doc 2는 네 단어 모두 df=1이라 IDF가 같고, 그래서 네 값이 모두 0.5입니다.
- doc 4에서 `astronauts`, `watched`(df=1)가 `baseball`, `game`(df=2)보다, 그리고 `space`(df=3)보다 큽니다. 이 문서를 가장 잘 구별하는 단어가 가장 큰 가중치를 받았습니다.

이 계산이 sklearn과 같은지 미리 확인해 봅니다(자세한 검증은 14장).

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer


def identity(doc):
    return doc


sk = TfidfVectorizer(analyzer=identity, lowercase=False).fit_transform(FILTERED).toarray()
print("sklearn과 최대 오차:", f"{np.abs(sk - TFIDF).max():.2e}")

sklearn과 최대 오차: 0.00e+00


### 새 문서에 적용하기
학습한 IDF로 **새 문서**(검색어 등)를 변환할 때도 같은 식을 씁니다. 반복이 있는 문서 `"space space rocket"`으로 raw TF와 sublinear TF를 비교합니다.

In [11]:
new_doc = ["space", "space", "rocket"]
c = np.array([new_doc.count(w) for w in VOCAB], dtype=float)
raw = c * idf
log_tf = np.zeros_like(c)
log_tf[c > 0] = 1 + np.log(c[c > 0])
sub = log_tf * idf
for name, vec in [("raw TF", raw), ("sublinear TF", sub)]:
    x = vec / np.linalg.norm(vec)
    print(f"{name:<13} space={x[VOCAB.index('space')]:.4f}  rocket={x[VOCAB.index('rocket')]:.4f}")

raw TF        space=0.8013  rocket=0.5982
sublinear TF  space=0.7500  rocket=0.6614


raw TF에서는 `space`가 두 번 나온 효과로 `rocket`보다 큽니다. sublinear TF에서는 반복 효과가 줄어 IDF가 큰 `rocket`이 앞섭니다.

## 📌 README에 넣을 수식 정리

문서 수를 $n$, 단어 $t$가 나온 문서 수를 $\mathrm{df}(t)$, 문서 $d$에서 $t$가 나온 횟수를 $f_{t,d}$라 하면:

$$
\mathrm{TF}(t,d) = f_{t,d}
\qquad
\mathrm{IDF}(t) = \ln\frac{1+n}{1+\mathrm{df}(t)} + 1
\qquad
\mathbf{x}_d = \frac{\mathrm{TF}(\cdot,d)\odot\mathrm{IDF}}{\lVert \mathrm{TF}(\cdot,d)\odot\mathrm{IDF}\rVert_2}
$$

- **TF**: 문서 안에서 자주 나온 단어일수록 그 문서를 잘 설명한다.
- **IDF**: 많은 문서에 나오는 단어일수록 문서를 구별하지 못하므로 가중치를 낮춘다. 로그로 차이를 완만하게 하고, smooth의 `+1`은 0으로 나누기를 막고, 끝의 `+1`은 모든 문서에 나온 단어도 0이 되지 않게 한다.
- **L2 정규화**: 문서 길이의 영향을 없애고 단어 구성만 비교한다.
- 변형: sublinear TF $1 + \ln f_{t,d}$ (sklearn `sublinear_tf=True`), no-smooth IDF $\ln\frac{n}{\mathrm{df}(t)} + 1$.

> 마크다운 셀을 더블클릭하면 수식의 LaTeX 원본을 볼 수 있습니다. README에 그대로 복사해 쓸 수 있습니다. GitHub의 README는 `$...$`, `$$...$$` 수식을 표시합니다.

## 🎯 학습 목표 ② 대비 Q&A

| 질문 | 답의 핵심 |
|---|---|
| TF는 무엇을 재나요? | 한 문서 안에서 단어가 나온 횟수. 그 문서에서 얼마나 중요한지 |
| IDF는 무엇을 재나요? | 단어가 얼마나 드문가. 많은 문서에 나올수록 작아진다 |
| 왜 둘을 곱하나요? | "이 문서에는 자주, 다른 문서에는 드물게" 나오는 단어에 큰 가중치를 주기 위해 |
| IDF에 왜 로그를 쓰나요? | 비율 차이가 너무 커서 희귀 단어 하나가 벡터를 지배하는 것을 막기 위해 |
| smooth의 +1은요? | 가상의 문서 하나를 더해 0으로 나누기를 막는다 |
| 끝의 +1은요? | 모든 문서에 나온 단어의 가중치가 0이 되지 않게 한다 |
| 정규화는 왜 하나요? | 긴 문서가 유리해지는 것을 막고, 내적을 코사인 유사도로 만들기 위해 |

## ✍️ 실습: doc 1 손계산

doc 1(`nasa launched space telescope`)의 TF-IDF를 손으로 계산하세요.
1. 네 단어의 df와 smooth IDF ($n = 5$)
2. TF×IDF 벡터와 그 L2 노름
3. 정규화된 값 네 개

종이로 계산한 뒤, 코드로 계산한 `TFIDF[1]`과 비교하세요. (힌트: $\ln 3 \approx 1.0986$, $\ln 2 \approx 0.6931$, $\ln 1.5 \approx 0.4055$)

In [12]:
# 여기에 작성하세요

**정답**

In [13]:
words = ["launched", "nasa", "space", "telescope"]
hand_idf = {"launched": np.log(6 / 3) + 1, "nasa": np.log(6 / 2) + 1,
            "space": np.log(6 / 4) + 1, "telescope": np.log(6 / 2) + 1}   # df = 2, 1, 3, 1
v = np.array([hand_idf[w] for w in words])                                 # TF가 모두 1
norm = np.sqrt((v ** 2).sum())
print("IDF     :", dict(zip(words, v.round(4))))
print(f"노름    : {norm:.4f}")
print("정규화  :", dict(zip(words, (v / norm).round(4))))
print("코드 결과와 같은가:", np.allclose(v / norm, [TFIDF[1, VOCAB.index(w)] for w in words]))

IDF     : {'launched': np.float64(1.6931), 'nasa': np.float64(2.0986), 'space': np.float64(1.4055), 'telescope': np.float64(2.0986)}
노름    : 3.6946
정규화  : {'launched': np.float64(0.4583), 'nasa': np.float64(0.568), 'space': np.float64(0.3804), 'telescope': np.float64(0.568)}
코드 결과와 같은가: True


## 🧐 퀴즈

**Q1.** 단어 A는 문서 하나에만 3번, 단어 B는 모든 문서에 한 번씩 나옵니다. smooth IDF에서 B의 IDF는 얼마이고, A와 B 중 어느 쪽이 그 문서에서 더 큰 TF-IDF를 받나요?
<details><summary>▶ 정답 보기</summary>

B는 $\mathrm{df} = n$이라 $\ln\frac{1+n}{1+n} + 1 = 1$입니다. A는 TF도 크고(3) IDF도 크므로($\ln\frac{1+n}{2} + 1 > 1$) A가 훨씬 큽니다.
</details>

**Q2.** 정규화 전에 TF를 문서 길이로 나누든 나누지 않든 최종 결과가 같은 이유는?
<details><summary>▶ 정답 보기</summary>

문서 길이로 나누는 것은 벡터 전체에 같은 수를 곱하는 것이라 방향이 바뀌지 않고, L2 정규화는 방향만 남기기 때문입니다.
</details>

**Q3.** smooth IDF의 분자·분모에 1을 더하는 것을 "가상의 문서"로 설명해 보세요.
<details><summary>▶ 정답 보기</summary>

모든 단어를 정확히 한 번씩 포함하는 문서가 하나 더 있다고 가정하면, 문서 수는 $n + 1$, 각 단어의 df는 $\mathrm{df} + 1$이 됩니다. 그래서 df가 0인 단어도 0으로 나누지 않습니다.
</details>

**Q4.** IDF 식에서 맨 끝의 +1을 빼면 어떤 단어가 어떻게 되나요?
<details><summary>▶ 정답 보기</summary>

모든 문서에 나오는 단어의 IDF가 $\ln 1 = 0$이 되어 TF-IDF 벡터에서 완전히 사라집니다.
</details>

**Q5.** sublinear TF는 어떤 문서에서 차이를 만드나요?
<details><summary>▶ 정답 보기</summary>

같은 단어가 여러 번 반복되는 문서입니다. 모든 단어가 한 번씩만 나오는 문서에서는 $1 + \ln 1 = 1$이라 raw TF와 같습니다.
</details>

## ✅ 체크포인트
- [ ] TF, IDF, TF-IDF, L2 정규화 수식을 보지 않고 쓸 수 있다.
- [ ] smooth IDF의 +1 세 개의 역할을 각각 설명할 수 있다.
- [ ] 미니 코퍼스 한 문서의 TF-IDF를 손으로 계산할 수 있다.
- [ ] raw TF와 sublinear TF가 다른 결과를 내는 예를 들 수 있다.

## 🔗 과제 연결과 다음 장
- 수식 정리는 README "구현 요약: TF-IDF 수식"에 씁니다(학습 목표 ②).
- 손계산 값(예: doc 0 = rocket 0.6902, launched 0.5568, space 0.4622)은 13장에서 구현을 테스트하는 정답이 됩니다.
- **13장. NumPy로 TF-IDF 구현하기**에서 이 계산을 재사용 가능한 클래스로 만듭니다.